# AZR Autopilot — Free LLM Server

This notebook launches a **free LLM** on Google Colab's GPU.

**No registration needed. No API keys. Just click Run all.**

### Quick start / Быстрый старт:
1. Click **Runtime → Run all** (Ctrl+F9)
2. Wait ~3-5 min for setup
3. A green box with your **URL** will appear — copy it
4. Paste it into the **Endpoint** field in the Autopilot tab

---

1. Нажмите **Runtime → Run all** (Ctrl+F9)
2. Подождите ~3-5 мин
3. Появится зелёная рамка с **URL** — скопируйте его
4. Вставьте в поле **Endpoint** во вкладке Автопилот

---
## Step 1/4 — Install Ollama & Cloudflare Tunnel

In [ ]:
%%capture
# Install Ollama
!curl -fsSL https://ollama.com/install.sh | sh
# Install cloudflared (free tunnel, no registration needed)
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared
print('OK')

## Step 2/4 — Start Ollama server

In [ ]:
import subprocess, time, os
os.environ['OLLAMA_HOST'] = '0.0.0.0:11434'
subprocess.Popen(['ollama', 'serve'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(3)
print('Ollama server started')

## Step 3/4 — Download model

Pick a model from the dropdown on the right. `llama3.2:3b` is recommended.

In [ ]:
#@title Select model { display-mode: "form" }
MODEL = 'llama3.2:3b' #@param ['llama3.2:3b', 'llama3.2:1b', 'gemma2:2b', 'qwen2.5:3b', 'qwen2.5:7b', 'mistral:7b']

print(f'Downloading {MODEL}...')
!ollama pull {MODEL}
print(f'\nModel {MODEL} ready!')

## Step 4/4 — Get your URL

Run this cell — your URL will appear in a big green box. **No registration needed!**

Запустите эту ячейку — URL появится в зелёной рамке. **Регистрация не нужна!**

In [ ]:
import subprocess, time, re, threading
from IPython.display import display, HTML, clear_output

# Start cloudflared tunnel in background
tunnel_url = None
tunnel_process = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', 'http://localhost:11434'],
    stdout=subprocess.PIPE, stderr=subprocess.PIPE
)

print('Starting tunnel...')

# Read stderr to find the URL
start_time = time.time()
while time.time() - start_time < 30:
    line = tunnel_process.stderr.readline().decode('utf-8', errors='ignore')
    match = re.search(r'(https://[a-zA-Z0-9\-]+\.trycloudflare\.com)', line)
    if match:
        tunnel_url = match.group(1)
        break

if tunnel_url:
    url = tunnel_url + '/v1'
    clear_output()
    display(HTML(f'''
    <div style="background:#065f46;border:3px solid #10b981;border-radius:12px;padding:24px;margin:16px 0;">
        <div style="color:#6ee7b7;font-size:14px;margin-bottom:4px;">COPY THIS URL / СКОПИРУЙТЕ ЭТОТ URL:</div>
        <div style="background:#000;border-radius:8px;padding:16px;margin:8px 0;">
            <code style="color:#4ade80;font-size:22px;user-select:all;cursor:text;">{url}</code>
        </div>
        <div style="color:#a7f3d0;font-size:13px;margin-top:8px;">Paste into <b>Endpoint</b> field / Вставьте в поле <b>Endpoint</b></div>
        <div style="color:#a7f3d0;font-size:13px;margin-top:4px;">Model / Модель: <code style="color:#4ade80;">{MODEL}</code></div>
        <div style="color:#6ee7b7;font-size:11px;margin-top:12px;">Keep this tab open! / Не закрывайте эту вкладку!</div>
    </div>
    '''))
    print(f'Endpoint: {url}')
    print(f'Model: {MODEL}')
else:
    display(HTML('<div style="background:#dc2626;color:#fff;padding:16px;border-radius:8px;font-size:16px;">'
        '<b>Failed to create tunnel.</b> Try running this cell again.'
        '</div>'))

## Keep alive

This cell keeps Colab from disconnecting. **Don't stop it** while using Autopilot.

Эта ячейка не даёт Colab отключиться. **Не останавливайте** пока используете Автопилот.

In [ ]:
import time
from IPython.display import display, HTML
display(HTML('<div style="color:#9ca3af;">Session active / Сессия активна</div>'))
i = 0
while True:
    time.sleep(60)
    i += 1
    print(f'\r  Active: {i} min', end='')